# Simulating a blue-detuned MOT for RaFOH

In [1]:
using Pkg
Pkg.activate("/Users/jose/Documents/Works/MIT/RaX/Simu/Molecule-Sims/")

  Activating project at `~/Documents/Works/MIT/RaX/Simu/Molecule-Sims`


In [2]:
# Import the primary packages required for the notebook
using
    Revise,
    QuantumStates,            # for calculating molecular structure
    OpticalBlochEquations,    # for solving optical Bloch equations
    DataFrames,
    CSV,
    Plots,
    PlotlyJS,
    UnitsToValue              # for numerical values
;

### Create Hamiltonian for the $X^2\Sigma^+(000, N=1)$ ground state

We first create a Hamiltonian for the $N=1$ state of the ground electronic $X^2\Sigma^+(000)$ state. Creating a Hamiltonian requiries defining three distinct objects:
1) A basis, accomplished using the function `enumerate_states`, which takes a basis state type as its first argument and appropriate bounds for the quantum numbers. Note that we let the basis include states of $N \in [0,3]$, not just $N=1$, since we could have matrix elements connecting the $N=1$ states to other rotational states. Such matrix elements are only included in the Hamiltonian provided that the basis includes all associated basis states. 
2) An operator for the Hamiltonian based on matrix elements that have been defined for the given basis state type. The syntax is to write the operator as a sum of terms of the form [parameter symbol] * [matrix operator], e.g., `BX * Rotation`.
3) A set of parameters using the macro `QuantumStates.@params` (note that other packages export a `@params` macro, hence the inclusion of the package name; normally we could just write `@params`). These parameters must correspond one-to-one with the parameter symbols used in the definition of the operator.

Additional terms can later be added to the Hamiltonian object using the `add_to_H` function, see below for an example.

Finally, the Hamiltonian is evaluated using `evaluate!` and then solved using `QuantumStates.solve!` (other packages export a `solve!` function, similarly to `@params` and the package name is therefore included in the function call).

In [3]:
QN_bounds = (
    S = 1/2,
    I = 1/2,
    Λ = 0,
    N = 0:3
)
X_state_basis = enumerate_states(HundsCaseB_LinearMolecule, QN_bounds)

X_state_operator = :(
    BX * Rotation +                     #
    DX * RotationDistortion +
    γX * SpinRotation +                 # Spin-rotation interaction
    bFX * Hyperfine_IS +
    cX * (Hyperfine_Dipolar/3)
)

# Now we have to plug in the constants for RaF:
# Taken from: https://www.nature.com/articles/s41567-023-02296-w
# If the results are being taken in

X_state_parameters = QuantumStates.@params begin
    # BX = 0.191985* c * 1e2
    BX = 5755.56e6
    # DX = 1.405 * (1e-7) * c * 1e2
    DX = 0.00420e6
    γX = 0.00585* c * 1e2
    # γX = 175.38e6
    bFX = 96.3e6
    cX = 19e6
end

X_state_ham = Hamiltonian(basis=X_state_basis, operator=X_state_operator, parameters=X_state_parameters)

B = 10.0 * 1e-4 # 10 Gauss

Zeeman_x(state, state′) = (Zeeman(state, state′,-1) - Zeeman(state, state′,1))/sqrt(2)
Zeeman_y(state, state′) = im*(Zeeman(state, state′,-1) + Zeeman(state, state′,1))/sqrt(2)
Zeeman_z(state, state′) = Zeeman(state, state′, 0)

_μB = (μB / h) * 1e-4
X_state_ham = add_to_H(X_state_ham, :B_x, (gS * _μB * 1e-6) * Zeeman_x)
X_state_ham = add_to_H(X_state_ham, :B_y, (gS * _μB * 1e-6) * Zeeman_y)
X_state_ham = add_to_H(X_state_ham, :B_z, (gS * _μB * 1e-6) * Zeeman_z)
X_state_ham.parameters.B_x = 0.
X_state_ham.parameters.B_y = B * 1/sqrt(2)
X_state_ham.parameters.B_z = B * 1/sqrt(2)

evaluate!(X_state_ham)
QuantumStates.solve!(X_state_ham)

;

### Create Hamiltonian for the $A^2\Pi_{1/2}(000, J=1/2+)$ state

We similarly define a Hamiltonian for the excited electronic state.

In [4]:
QN_bounds = (
    S = 1/2,
    I = 1/2,
    Λ = (-1,1),
    J = 1/2:3/2 # ?
)
A_state_basis = enumerate_states(HundsCaseA_LinearMolecule, QN_bounds)

A_state_operator = :(
    T_A * DiagonalOperator +
    Be_A * Rotation +
    Aso_A * SpinOrbit +
    q_A * ΛDoubling_q +
    p_A * ΛDoubling_p2q + q_A * (2ΛDoubling_p2q) +
    a * Hyperfine_IL +
    d * Hyperfine_Dipolar_d
)

# Spectroscopic constants for CaOH, A state
A_state_parameters = QuantumStates.@params begin
    T_A = (2067.6/2+13284.427) * c * 1e2 # Diagonal constant (electrion zero point energy)
    Be_A = 0.191 * c * 1e2# Rotational constant
    Aso_A = 2067.6 * c * 1e2# 1.405 * (1e-7) # A spin-orbit constant
    # Lambda doubling
    p_A = −0.41071 * c * 1e2
    q_A = 0
    a = 19/2 * 1e6
    d = -9e6
end

A_state_ham = Hamiltonian(basis=A_state_basis, operator=A_state_operator, parameters=A_state_parameters)
evaluate!(A_state_ham)
QuantumStates.solve!(A_state_ham)
;

### Convert relevant $A$ states ($J=1/2+$) to a Hund's case (b) basis

We used a Hund's case (a) basis to define the Hamiltonian for the excited $A$ state. In order to compute transition dipole moments with the ground state we wish to convert these states to a Hund's case (b). We can accomplish this by simply defining a new basis that has all the Hund's case (b) states required for this conversion, and then using the function `convert_basis`. 

In [5]:
function get_basis_elems(state)
    coeffs = abs.(state.coeffs) .^ 2
    max_idx = argmax(coeffs)
    return state.basis[max_idx]
end

function extract_quantum_numbers(state)
    bass = get_basis_elems(state)
    dict_info = Dict(:N => bass.N, :J => bass.J, :F => bass.F, :M => bass.M)
    return dict_info
end

function get_j_from_state(state)
    bass = get_basis_elems(state)
    return bass.J
end

function search_state_by_quantum_numbers(states, N, J, F, M)
    for state in states
        qn = extract_quantum_numbers(state)
        if qn[:N] == N && qn[:J] == J && qn[:F] == F && qn[:M] == M
            return state
        end
    end
end

search_state_by_quantum_numbers (generic function with 1 method)

In [6]:
A_J12 = [state for state in A_state_ham.states if get_j_from_state(state) == 1/2]
A_state_J12_pos_parity_states = A_J12[5:8]

QN_bounds = (
    S = 1/2,
    I = 1/2,
    Λ = (-1,1),
    N = 0:3 # ?
)
A_state_caseB_basis = enumerate_states(HundsCaseB_LinearMolecule, QN_bounds)

ground_states = X_state_ham.states[5:16] # This is dropping all the terms with N = 0
excited_states = convert_basis(A_state_J12_pos_parity_states, A_state_caseB_basis)

states = [ground_states; excited_states]
;

In [7]:
# # Put energy data in a DataFrame and plot it using `PlotlyJS`.
# using PlotlyJS
# Aenergies = [energy(st) for st in excited_states]
# Xenergies = [energy(st) for st in all_grounds]

# ground_quantum_numbers = [extract_quantum_numbers(st) for st in all_grounds]
# excited_quantum_numbers = [extract_quantum_numbers(st) for st in excited_states]

# # For all_grounds
# X_Js = [qn[:J] for qn in ground_quantum_numbers]
# X_Ms = [qn[:M] for qn in ground_quantum_numbers]
# X_Fs = [qn[:F] for qn in ground_quantum_numbers]
# X_Ns = [qn[:N] for qn in ground_quantum_numbers]

# # For excited_states
# A_Js = [qn[:J] for qn in excited_quantum_numbers]
# A_Ms = [qn[:M] for qn in excited_quantum_numbers]
# A_Fs = [qn[:F] for qn in excited_quantum_numbers]
# A_Ns = [qn[:N] for qn in excited_quantum_numbers]

# # Create a DataFrame
# using DataFrames

# df = DataFrame(
#     J = [A_Js; X_Js],
#     M = [A_Ms; X_Ms],
#     F = [A_Fs; X_Fs],
#     N = [A_Ns; X_Ns],
#     energy = [Aenergies; Xenergies],
#     state = [["A" for _ in Aenergies]; ["X" for _ in Xenergies]]
# )

# # Sort the DataFrame by energy and then M
# sort!(df, [:energy, :M, :F, :J, :N])

In [8]:
# # Add a different marker for different M values and label each point by the quantum numbers
# PlotlyJS.plot(
#     scatter(
#         x = df.N[df.state .== "X"],
#         y = df.energy[df.state .== "X"],
#         mode = "markers",
#         marker = attr(
#             size = 10,
#             color = df.M,
#             colorscale = "Viridis",
#             colorbar = attr(title = "M"),
#             showscale = true
#         ),
#         text = ["J: $(df.J[i]), M: $(df.M[i]), F: $(df.F[i]), N: $(df.N[i])" for i in 1:length(df.energy)],
#         hoverinfo = "text",
#         name = "A and X states",
#         type = "scatter"
#     ),
# )

### Compute dipole moments between states

The optical Bloch equations (OBE) solver requires a matrix `d` for the transition dipole moments (TDMs) among all states relevant to the simulation. In our case we have 12 ground states and 4 excited states, so $d$ will have dimensions $16 \times 16 \times 3$. TDMs between two sets of states can be computed by first getting the TDMs between the two bases used to define the two sets of states (using `get_tdms_two_basis`), and then using those TDMs between basis states to compute the TDMs between the sets of states (using `tdms_between_states!`).

In [9]:
d = zeros(ComplexF64, 16, 16, 3)
d_ge = zeros(ComplexF64, 12, 4, 3)

basis_tdms = get_tdms_two_bases(X_state_ham.basis, A_state_caseB_basis, TDM)
tdms_between_states!(d_ge, basis_tdms, ground_states, excited_states)
d[1:12, 13:16, :] .= d_ge
;

### Zeeman plot for the $X^2\Sigma^+(N=1)$ state in RaF

In [10]:
# A few constants used for the simulation
λ = 1e-2/13324 # Wavelength of light in meters
Γ = 2π/(34e-9) # 34 ns
m = @with_unit 245 "u" # Mass of the molecule in atomic mass units
k = 2π / λ
;

In [11]:
# Get the basis element with the maximum coefficient from the state
function get_max_coeff_basis_element(state)
    coeffs = abs.(state.coeffs) .^ 2
    max_idx = argmax(coeffs)
    return state.basis[max_idx]
end

# Extract quantum numbers from the basis element with the maximum coefficient
function extract_quantum_numbers_from_state(state)
    basis_elem = get_max_coeff_basis_element(state)
    quantum_numbers = Dict(:N => basis_elem.N, :J => basis_elem.J, :F => basis_elem.F, :M => basis_elem.M)
    return quantum_numbers
end

# Get the J quantum number from the basis element with the maximum coefficient
function get_J_quantum_number(state)
    basis_elem = get_max_coeff_basis_element(state)
    return basis_elem.J
end

# Search for a state in a list of states by specific quantum numbers
function find_state_by_quantum_numbers(states, N, J, F, M)
    for state in states
        qn = extract_quantum_numbers_from_state(state)
        if qn[:N] == N && qn[:J] == J && qn[:F] == F && qn[:M] == M
            return state
        end
    end
    error("State with quantum numbers N=$N, J=$J, F=$F, M=$M not found.")
end
Aenergies = [energy(st) for st in excited_states]
Xenergies = [energy(st) for st in all_grounds]

ground_quantum_numbers = [extract_quantum_numbers_from_state(st) for st in all_grounds]
excited_quantum_numbers = [extract_quantum_numbers_from_state(st) for st in excited_states]

X_Js = [qn[:J] for qn in ground_quantum_numbers]
X_Ms = [qn[:M] for qn in ground_quantum_numbers]
X_Fs = [qn[:F] for qn in ground_quantum_numbers]
X_Ns = [qn[:N] for qn in ground_quantum_numbers]
A_Js = [qn[:J] for qn in excited_quantum_numbers]
A_Ms = [qn[:M] for qn in excited_quantum_numbers]
A_Fs = [qn[:F] for qn in excited_quantum_numbers]
A_Ns = [qn[:N] for qn in excited_quantum_numbers]

df = DataFrame(
    J = [A_Js; X_Js],
    M = [A_Ms; X_Ms],
    F = [A_Fs; X_Fs],
    N = [A_Ns; X_Ns],
    energy = [Aenergies; Xenergies],
    state = [["A" for _ in Aenergies]; ["X" for _ in Xenergies]]
)

# Sort the DataFrame by energy and then M
sort!(df, [:energy, :M, :F, :J, :N]);
# PlotlyJS.plot(
#     PlotlyJS.scatter(
#         x = df.N[df.state .== "X"],
#         y = df.energy[df.state .== "X"],
#         mode = "markers",
#         marker = attr(
#             size = 10,
#             color = df.M,
#             colorscale = "Viridis",
#             colorbar = attr(title = "M"),
#             showscale = true
#         ),
#         text = ["J: $(df.J[i]), M: $(df.M[i]), F: $(df.F[i]), N: $(df.N[i])" for i in 1:length(df.energy)],
#         hoverinfo = "text",
#         name = "A and X states",
#         type = "scatter"
#     ),
# )

UndefVarError: UndefVarError: `all_grounds` not defined

# Setup the laser scheme

source states:

 - All N=1 states of the $X^2\Sigma^+(000)$ state

Target states:

 - State $J=1/2+$ states of the $A^2\Pi_{1/2}(000)$ with $N=1$

In [12]:
# source states
input_qn = [elem for elem in ground_quantum_numbers if elem[:N] == 1 && elem[:J] == 1/2]#[1]
input_states = [search_state_by_quantum_numbers(all_grounds, elem[:N], elem[:J], elem[:F], elem[:M]) for elem in input_qn]
input_energies = [energy(st) for st in input_states]

# Target states
a_qns =[extract_quantum_numbers(st) for st in excited_states]
output_qn = [elem for elem in excited_quantum_numbers if elem[:N] == 1 && elem[:J] == 1/2 && elem[:F] == 0]#[1]
target_energy = [energy(st) for st in excited_states if extract_quantum_numbers(st) == output_qn[1]];

UndefVarError: UndefVarError: `ground_quantum_numbers` not defined

In [13]:
delta_state_energies = target_energy .- input_energies[1]

UndefVarError: UndefVarError: `target_energy` not defined

We need to define the lasers that we wish to consider for our simulation. This is accomplished using the type `Field`, which takes a (unit!) $k$ vector, a function to set the polarization (defined in the spherical basis), a laser frequency (in angular units), and a saturation parameter. 

In [14]:
s_func(s) = (r,t) -> s
s = s_func(10.0)

A_energy = target_energy[1]

δs = [+2Γ for _ in input_states]
polarizations = [σ⁺ for _ in input_states]
ωs = [2π * (A_energy - input_energies[i]) + δs[i] for i in 1:length(input_states)]
ϵ(ϵ_val) = t -> ϵ_val # polarization as a function of time


UndefVarError: UndefVarError: `target_energy` not defined

In [15]:

# x axis
k̂ = +x̂; ϵs = [ϵ(rotate_pol(polarizations[j], ŷ)) for j in 1:length(polarizations)]
lasers_plus_x = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]
k̂ = -x̂; ϵs = [ϵ(rotate_pol(polarizations[j], ŷ)) for j in 1:length(polarizations)]
lasers_minus_x = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

# # y axis
k̂ = +ŷ; ϵs = [ϵ(rotate_pol(polarizations[j], +ẑ)) for j in 1:length(polarizations)]
lasers_plus_y = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]
k̂ = -ŷ; ϵs = [ϵ(rotate_pol(polarizations[j], +ẑ)) for j in 1:length(polarizations)]
lasers_minus_y = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

# # z axis
# k̂ = +ẑ; ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]
# lasers_plus_z = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]
# k̂ = -ẑ; ϵs = [ϵ(rotate_pol(polarizations[j], k̂)) for j in 1:length(polarizations)]
# lasers_minus_z = [Field(k̂, ϵ, ω, s) for (ϵ, j, ω) in zip(ϵs, polarizations, ωs)]

lasers = [lasers_plus_x; lasers_minus_x; lasers_plus_y; lasers_minus_y;]# lasers_plus_z; lasers_minus_z]
;

UndefVarError: UndefVarError: `polarizations` not defined

The OBE solver requires a set of parameters `p`, which are defined using the function `obe`. Most of these arguments should be self-explanatory, but a few require special attention:
* `freq_res` _(type: Float64)_: The resolution used when rounding frequencies (and the velocity) in order to ensure that the OBEs reach a quasi-periodic steady state. Defined in units of Γ and typically set to 1e-1 (or 1e-2 for low velocities).
* `extra_p` _(type: NamedTuple)_: Allows the user to define a set of "extra" parameters that may be used during the simulation. This is mostly relevant in cases where we wish to solve the OBEs several times and we have some parameter that is changing between solves, e.g., we may need to update our TDMs if we want to solve the OBEs for a range of magnetic fields. In that case it's helpful to ensure that the solver has access to the underlying Hamiltonian, as well as any data structures we've defined to help in updating the TDMs.

In [16]:
# Set initial conditions
particle = OpticalBlochEquations.Particle()
ρ0 = zeros(ComplexF64, length(states), length(states))
# names = [
# "M = $(st.basis[i].M), J = $(st.basis[i].J), F = $(st.basis[i].F))" for (i, st) in enumerate(states)]
names = [extract_quantum_numbers(st) for st in states]
string_names = ["|$(qn[:J]),  $(qn[:F]), $(qn[:M])>" for qn in names]
ρ0[1,1] = 1.0

freq_res = 1e-2
p = obe(ρ0, particle, states, lasers, d, true, true; λ=λ, Γ=Γ, freq_res=freq_res)
;

UndefVarError: UndefVarError: `lasers` not defined

After defining `p`, we're free to update any of its parameters before solving the OBEs. For example, we can update the initial position `r0` to $\mathbf{r}_0 = (0,0,0.5)~k^{-1}$ and the velocity `v` to $\mathbf{v} = (0,0,1)~\text{m/s}$ as shown below. Note that position and velocity need to have units of 1/k and Γ/k in the simulation, so since we defined the velocity in SI units it must be divided by Γ/k. We also need to round the velocity again (using `round_vel`) after updating it (this rounding is automatically performed inside `obe` when `p` is first defined but must be performed again if `v` is changed).

In [17]:
p.r0 = (0., 0., 0.5)
p.v = (0., 0., 1.0) ./ (Γ / k)
p.v = round_vel(p.v, p.freq_res)
;

UndefVarError: UndefVarError: `p` not defined

Finally, we define a time span for the solver (here in terms of the period defined by `freq_res`, which is given by `p.period`). The actual solving is performed using the `DifferentialEquations` package in Julia, so we load this here as well. This package requires a `prob` variable of type `ODEProblem`. The cell below shows the syntax for how this variable is created for the OBE solver.

After solving, we can plot the populations as a function of time, and the force averaged over one period can be found in `prob.p.force_last_period`. Note that the force output in the simulation has units ħkΓ.

In [18]:
using DifferentialEquations
using Plots
t_end = 50p.period+1
tspan = (0., t_end)
prob = ODEProblem(ρ!, p.ρ0_vec, tspan, p)
times = range(0, t_end, 1_00)

cb = PeriodicCallback(reset_force!, p.period)
@time sol = DifferentialEquations.solve(prob, DP5(), callback=cb, reltol=1e-3, saveat=times)

# Print the force
println()
print("Force (10³ m/s): ", prob.p.force_last_period * (1e-3 * ħ * k * Γ / m))

using Plots
plot_us = sol.u
plot_ts = sol.t

n_states = size(p.ρ_soa, 1)
Plots.plot(
    ylim=(-0.1, 1.6),
)

for i in 1:n_states
    state_idx = n_states*(i-1) + i # For the diagonal elements
    Plots.plot!(plot_ts, [real(u[state_idx]) for u in plot_us], label=string_names[i])
end


# # plot!()
# offset = 0
# vline!([sol.t[end] - prob.p.period - offset, sol.t[end] - offset], color="red", linestyle=:dash)
# Add the correct axis labels
Plots.plot!(
    xlabel="Time (s)",
    ylabel="Population"
)
# Add the legends to each plot


UndefVarError: UndefVarError: `p` not defined

### Force versus velocity

Creating a "force profile" (i.e., the force as a function of some variable) can be performed by combining the previous section with the function `force_scan_v3`. The function takes in a `prob` for the OBEs (which we already defined above) and a `scan_values_grid` (essentially a grid of all the values we wish to scan over). Note that `scan_values_grid` may be multidimensional, e.g., we may wish to scan over both position and velocity simultaneously. Indeed, a profile of force versus velocity typically involves averaging over a collection of different starting positions from $\mathbf{r} = (0,0,0)$ to $\mathbf{r} = (2\pi, 2\pi, 2\pi)$ (in units of 1/k), or $\mathbf{r} = (0,0,0)$ to $\mathbf{r} = (\lambda, \lambda, \lambda)$ (in SI units). This ensures that the force is averaged over all possible initial values of the combined laser field seen by the molecule.

Two additional arguments are also required for `force_scan_v3`:
* `prob_func!` _(prob, scan_params, i) -> prob_: Updates the initial `prob` for the `i`th value of the `scan_values_grid`.
* `output_func` _(p, sol) -> force_: Define the output of the solver after `prob` has been solved, typically the force.

In [19]:
using
    StaticArrays,
    RectiGrids,
    StatsBase

In [20]:
function prob_func!(prob, scan_values_grid, i)
    p = prob.p
    # Update velocity and position
    p.v .= (0, 0, scan_values_grid[i].v)
    p.v .= round_vel(p.v, p.freq_res)
    p.r0 .= scan_values_grid[i].r
    return prob
end
function output_func(p, sol)
    f = p.force_last_period
    return (f[1], 0, 0)
end
;

In [21]:
freq_res = 1e-1
p = obe(ρ0, particle, states, lasers, d, true, true; λ=λ, Γ=Γ, freq_res=freq_res)

t_end = 10p.period+1; tspan = (0., t_end)
prob = ODEProblem(ρ!, p.ρ0_vec, tspan, p, reltol=1e-3, save_on=false)

di = 2
rs = vcat([(n1/(di+1), n2/(di+1), n3/(di+1)) .* 2π for n1 ∈ 0:di, n2 ∈ 0:di, n3 ∈ 0:di]...)
vs = -7:0.5:7.0

scan_values = (r = rs, v = vs)
scan_values_grid = RectiGrids.grid(scan_values)
;

UndefVarError: UndefVarError: `lasers` not defined

In [22]:
@time forces, populations = force_scan_v2(prob, scan_values_grid, prob_func!, output_func);

UndefVarError: UndefVarError: `prob` not defined

In [23]:
averaged_forces = Float64[]
@time for (i,v) ∈ enumerate(vs)
    idxs = [j for (j,x) ∈ enumerate(scan_values_grid) if x.v == v]
    push!(averaged_forces, mean([f[3] for f in forces[idxs]]))
end

UndefVarError: UndefVarError: `vs` not defined

In [24]:
Plots.plot(vs, (1e-3 * ħ * k * Γ / m) .* averaged_forces,
    xlabel="Velocity (m/s)",
    ylabel="a (10³ m/s²) - x direction",
    framestyle=:box,
    linewidth=2.5,
    legend=nothing
    )

UndefVarError: UndefVarError: `vs` not defined